# 05 · Araç Değişken: Eğitimin Gerçek Getirisi

Daha yetenekli insanlar hem daha uzun okur hem daha çok kazanır. Yetenek gözlenmediği için EKK, eğitimin getirisini **olduğundan büyük** tahmin eder (yetenek yanlılığı). Çözüm: eğitimi etkileyen ama ücreti **yalnızca eğitim aracılığıyla** etkileyen bir **araç değişken**.

Bu derste Card (1995) yaklaşımını izliyoruz: büyüdüğün yerin üniversiteye uzaklığı ve kardeş sayısı.

**Öğreneceklerin**
1. 2SLS ile içsel değişkeni tahmin etmek
2. Zayıf araç, içsellik ve aşırı tanımlama testlerini okumak
3. EKK ve 2SLS sonuçlarını karşılaştırıp raporlamak

> ⚠️ Kullanılan veri **simüle edilmiştir**: gerçek getiri 0,075.

In [1]:
import ekonometri as ek

veri = ek.veri.yukle("getiri")
print(ek.veri.aciklama("getiri"))

📁 getiri — Eğitimin getirisi (araç değişken)
⚠️ Simüle veri: gerçek gözlem değildir, eğitim amaçlıdır.

Konu: İçsellik, araç değişken (2SLS), zayıf araç, Wu-Hausman içsellik ve Sargan aşırı tanımlama testleri

Değişkenler:
  • ucret: Aylık ücret (TL)
  • egitim: Eğitim süresi (yıl) — içsel: gözlenmeyen yetenekle ilişkili
  • tecrube: İş tecrübesi (yıl)
  • cinsiyet: Erkek / Kadın
  • yerlesim: Kent / Kır (büyüdüğü yer)
  • uzaklik: Büyüdüğü yerin en yakın üniversiteye uzaklığı (km) — araç
  • kardes: Kardeş sayısı — araç

Veriyi üreten gerçek model (cevap anahtarı):
  log(ucret) = 9 + 0,075·egitim + 0,03·tecrube − 0,0005·tecrube² − 0,12·Kadın + 0,06·Kent + 0,12·yetenek + u;  yetenek gözlenmez ve eğitimi de artırır, bu yüzden EKK eğitimin getirisini olduğundan yüksek tahmin eder.

Örnek:
  ek.iv2sls("np.log(ucret) ~ tecrube + I(tecrube**2) + cinsiyet + yerlesim + [egitim ~ uzaklik + kardes]", veri)


## 1. EKK: yanlı tahmin

In [2]:
ekk = ek.ekk("np.log(ucret) ~ egitim + tecrube + I(tecrube**2) + cinsiyet + yerlesim", veri)
ekk.katsayilar["egitim"]

np.float64(0.10063942795781398)

## 2. 2SLS

İçsel değişken ve araçları formülde köşeli parantezle yazılır: `[egitim ~ uzaklik + kardes]`.

In [3]:
iv = ek.iv2sls("np.log(ucret) ~ tecrube + I(tecrube**2) + cinsiyet + yerlesim + [egitim ~ uzaklik + kardes]", veri,
               birimler={"egitim": "yıl", "tecrube": "yıl"})
iv

══════════════════════════════════════════════════════════════════════════════
 2SLS (Araç Değişken) Sonuçları — bağımlı değişken: ln(ucret)
══════════════════════════════════════════════════════════════════════════════
 Gözlem sayısı: 2000       R²: 0,3733     Wald χ²(5) = 513,25 (p <0,0001)
 İçsel: egitim   Araçlar: uzaklik, kardes
 Standart hatalar: klasik
──────────────────────────────────────────────────────────────────────────────
 Değişken                         Katsayı   Std. hata        z         p
──────────────────────────────────────────────────────────────────────────────
 Sabit                             8,9823      0,0796   112,78   <0,0001 ***
 cinsiyet: Kadın                  -0,1038      0,0146    -7,11   <0,0001 ***
 yerlesim: Kent                    0,0645      0,0170     3,78    0,0002 ***
 tecrube                           0,0337      0,0036     9,25   <0,0001 ***
 tecrube²                       -0,000637    0,000117    -5,46   <0,0001 ***
 egitim ◂             

## 3. Tanı testleri

- **Birinci aşama F**: araçlar eğitimi yeterince güçlü açıklıyor mu? (F < 10 → zayıf araç) — Stata: `estat firststage`
- **Durbin ve Wu-Hausman**: eğitim gerçekten içsel mi? (içsel değilse EKK daha etkin) — Stata: `estat endogenous`
- **Sargan ve Basmann**: araç sayısı içsel değişken sayısından fazlaysa, araçlar birbiriyle tutarlı mı? — Stata: `estat overid`

Standart hatalar Stata `ivregress 2sls` varsayılanıyla (büyük örneklem, z istatistiği) hesaplanır; `small` seçeneği için `kucuk_orneklem=True`.

In [4]:
for t in iv.tanilar():
    print(t, end='\n\n')

✅ Birinci aşama F (egitim): istatistik = 259,185 [2, 1993], p < 0,0001
   H0: Dışlanmış araçların içsel değişken üzerindeki etkisi sıfırdır.
   F = 259,18 ≥ 10: araçlar güçlü (kısmi R² = 0,206).

⚠️ Durbin içsellik testi: istatistik = 14,942 [1], p = 0,0001
   H0: İçsel varsayılan değişken(ler) aslında dışsaldır (EKK tutarlıdır).
   p < 0,05: H0 reddedildi — değişken içsel; EKK tutarsız, 2SLS kullanılmalı.

⚠️ Wu-Hausman içsellik testi: istatistik = 15,002 [1, 1993], p = 0,0001
   H0: İçsel varsayılan değişken(ler) aslında dışsaldır (EKK tutarlıdır).
   p < 0,05: H0 reddedildi — değişken içsel; EKK tutarsız, 2SLS kullanılmalı.

✅ Sargan aşırı tanımlama testi: istatistik = 0,264 [1], p = 0,6074
   H0: Tüm araçlar geçerlidir (hata terimiyle ilişkisizdir).
   p ≥ 0,05: H0 reddedilemedi — aşırı tanımlama kısıtları geçerli; araçların geçersizliğine dair kanıt yok.

✅ Basmann aşırı tanımlama testi: istatistik = 0,263 [1], p = 0,6080
   H0: Tüm araçlar geçerlidir (hata terimiyle ilişkisizdir)

In [5]:
iv.yorumla()

📊 Yorum (2SLS) — 'ucret' (logaritması)

▸ Model geneli
  • İçsel değişken: 'egitim'; araçlar: 'uzaklik', 'kardes'.
  • Gözlem sayısı: 2.000 · standart hatalar: klasik.
  • Birinci aşama F (egitim): F = 259,18 ≥ 10: araçlar güçlü (kısmi R² = 0,206).
  • Durbin içsellik testi: p < 0,05: H0 reddedildi — değişken içsel; EKK tutarsız, 2SLS kullanılmalı.
  • Wu-Hausman içsellik testi: p < 0,05: H0 reddedildi — değişken içsel; EKK tutarsız, 2SLS kullanılmalı.
  • Sargan aşırı tanımlama testi: p ≥ 0,05: H0 reddedilemedi — aşırı tanımlama kısıtları geçerli; araçların geçersizliğine dair kanıt yok.
  • Basmann aşırı tanımlama testi: p ≥ 0,05: H0 reddedilemedi — araçların geçersizliğine dair kanıt yok.

▸ Katsayılar (diğer değişkenler sabitken)
  • 'cinsiyet' değişkeninde 'Kadın' grubunun 'ucret' değeri, referans grupla ('Erkek') karşılaştırıldığında ortalama %9,9 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'yerlesim' değişkeninde 'Kent' grubunun 'ucret' değeri, referans grupla ('Kır') karşılaştırıldığında ortalama %6,7 daha yüksek — %1 düzeyinde istatistiksel olarak anlamlı (p = 0,0002).
  • 'tecrube' değişkeninin etkisi doğrusal değil: önce artan sonra azalan (ters U) bir ilişki var. Etki, 'tecrube' ≈ 26,43 yıl civarında yön değiştiriyor (doğrusal terim p < 0,0001; kare terim p < 0,0001, %1 düzeyinde istatistiksel olarak anlamlı).
  • Ortalama düzeyde ('tecrube' = 15,13 yıl), 'tecrube' 1 yıl artınca 'ucret' yaklaşık %1,5 artıyor.
  • 'egitim' 1 yıl artınca 'ucret' ortalama %7,73 artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).

▸ Notlar
  • EKK tahmini 'egitim' için 0,1006, 2SLS tahmini 0,0745: EKK, içsellik nedeniyle etkiyi olduğundan büyük gösteriyor olabilir.
  • Bağımlı değişken logaritmik: katsayılar yüzde değişim olarak yorumlandı (%100·(eᵇ−1)).
  • 2SLS tahmininin nedensel yorumu, araçların geçerli olmasına (yalnızca içsel değişken aracılığıyla etki etmesine) bağlıdır; bu varsayım test edilemez, kuramla savunulmalıdır.

## 4. EKK ve 2SLS yan yana

In [6]:
ek.karsilastir(iv.ekk, iv, adlar=["EKK", "2SLS"])

Değişken,EKK,2SLS
cinsiyet: Kadın,"-0,1005***","-0,1038***"
,"(0,0144)","(0,0146)"
yerlesim: Kent,"0,0424***","0,0645***"
,"(0,0158)","(0,0170)"
tecrube,"0,0342***","0,0337***"
,"(0,0036)","(0,0036)"
tecrube²,"-0,000647***","-0,000637***"
,"(0,000115)","(0,000117)"
egitim,"0,1006***","0,0745***"
,"(0,0035)","(0,0077)"


Tam tez bölümü (Yöntem, birinci aşama tablosu, tanı testleri, karşılaştırma ve yorum): `iv.rapor("arac_degisken.docx")`

> **Önemli:** Araçların geçerliliği (dışlama kısıtı) istatistiksel olarak tam sınanamaz. Sargan testi yalnızca araçların *birbiriyle* tutarlılığına bakar; hepsi aynı yönde geçersizse yakalayamaz. Araç seçimini kuramla savunmak gerekir.
